# Notebook 1: Data Loading & Preprocessing
## AI-Generated Face Detection



**Dataset:** `Madushan996/real_fake_images` (Updated Distribution)
- ~3,000 Real images
- ~1,000 Stable Diffusion fakes
- ~1,000 Flux fakes
- ~1,000 Z-Image fakes


**Outputs:**
- Center-cropped face images (224×224 RGB)
- DCT frequency representations (224×224 single-channel)
- `manifest.csv` with paths, labels, generator info, and split assignments

## 1. Setup & Installation

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Create project directory on Drive
import os

PROJECT_DIR = '/content/drive/MyDrive/deepfake_detection'
RAW_DIR = '/content/raw_data'           # Temporary (Colab local, faster I/O)
PROCESSED_DIR = os.path.join(PROJECT_DIR, 'processed_dataset')

for d in [PROJECT_DIR, RAW_DIR, PROCESSED_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Project directory: {PROJECT_DIR}')
print(f'Raw data (temp):   {RAW_DIR}')
print(f'Processed output:  {PROCESSED_DIR}')

In [ ]:
%%capture
# Install required libraries
!pip install datasets huggingface_hub albumentations scipy pillow tqdm

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm
import cv2
import json
import shutil
import warnings
warnings.filterwarnings('ignore')

print('All imports successful.')

## 2. Download Dataset from HuggingFace

In [ ]:
from huggingface_hub import snapshot_download
from google.colab import userdata

# Retrieve HF_TOKEN from Colab Secrets
try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None
    print('Warning: HF_TOKEN not found in Secrets. Proceeding with unauthenticated request.')

# Download entire dataset repo to local storage (faster than Drive)
print('Downloading dataset from HuggingFace...')
print('This may take 5-10 minutes depending on connection speed.')

dataset_path = snapshot_download(
    repo_id='Madushan996/real_fake_images',
    repo_type='dataset',
    local_dir=os.path.join(RAW_DIR, 'real_fake_images'),
    local_dir_use_symlinks=False,
    token=hf_token
)

print(f'\nDataset downloaded to: {dataset_path}')

In [ ]:
# Verify dataset structure and count images
data_root = os.path.join(RAW_DIR, 'real_fake_images')

folders = {
    'real': os.path.join(data_root, 'real'),
    'fake_sd': os.path.join(data_root, 'fake_sd'),
    'fake_fx': os.path.join(data_root, 'fake_fx'),
    'fake_z': os.path.join(data_root, 'fake_z'),
}

IMG_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

counts = {}
for name, folder in folders.items():
    if os.path.exists(folder):
        files = [f for f in os.listdir(folder) if Path(f).suffix.lower() in IMG_EXTENSIONS]
        counts[name] = len(files)
    else:
        counts[name] = 0
        print(f'WARNING: Folder not found: {folder}')

print('\n--- Dataset Summary ---')
for name, count in counts.items():
    label = 'REAL' if name == 'real' else 'FAKE'
    print(f'  {name:10s} → {count:5d} images  [{label}]')

total_images = sum(counts.values())
real_count = counts.get('real', 0)
fake_count = total_images - real_count

print(f'  {"TOTAL":10s} → {total_images:5d} images')
print(f'  Real/Fake ratio: {real_count} / {fake_count}')

## 3. Center-Crop & Resize

All images in the dataset are already face-centric portraits (500×500).
We apply a **center square crop** and **resize to 224×224** for ViT input.
No face detection model is needed since every image contains a clear, centered face.

In [ ]:
TARGET_SIZE = 224  # Output crop size for ViT input

def center_crop_and_resize(image_path, target_size=TARGET_SIZE):
    """
    Center-crop to a square and resize to target_size.
    Handles both square and non-square inputs.

    Returns:
        np.ndarray (H, W, 3) RGB image, or None if image cannot be loaded.
    """
    try:
        img = Image.open(image_path).convert('RGB')
    except Exception:
        return None

    w, h = img.size

    # Center square crop
    crop_size = min(w, h)
    left = (w - crop_size) // 2
    top = (h - crop_size) // 2
    img = img.crop((left, top, left + crop_size, top + crop_size))

    # Resize to target
    img = img.resize((target_size, target_size), Image.LANCZOS)

    return np.array(img)

print('Center-crop function ready.')

In [ ]:
# Quick test: process one image from each folder
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for idx, (name, folder) in enumerate(folders.items()):
    if not os.path.exists(folder):
        continue
    files = sorted([f for f in os.listdir(folder) if Path(f).suffix.lower() in IMG_EXTENSIONS])
    if files:
        test_path = os.path.join(folder, files[0])
        crop = center_crop_and_resize(test_path)
        if crop is not None:
            axes[idx].imshow(crop)
            axes[idx].set_title(f'{name} ({crop.shape[0]}×{crop.shape[1]})')
        else:
            axes[idx].set_title(f'{name} (load failed)')
    axes[idx].axis('off')

plt.suptitle('Sample Crops (one per category)', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Process All Images

Center-crop and resize every image.

In [ ]:
# Create output directory structure
CROPS_DIR = os.path.join(PROCESSED_DIR, 'crops')

for name in folders.keys():
    os.makedirs(os.path.join(CROPS_DIR, name), exist_ok=True)

print(f'Output directory: {CROPS_DIR}')
print('Subdirectories created:', list(folders.keys()))

In [ ]:
# Process all images
records = []     # For manifest.csv
failures = []    # Track failed loads

for category, folder in folders.items():
    if not os.path.exists(folder):
        print(f'Skipping {category}: folder not found')
        continue

    files = sorted([f for f in os.listdir(folder) if Path(f).suffix.lower() in IMG_EXTENSIONS])
    label = 0 if category == 'real' else 1  # 0 = real, 1 = fake

    # Determine generator name
    generator_map = {
        'real': 'none',
        'fake_sd': 'stable_diffusion',
        'fake_fx': 'flux',
        'fake_z': 'z_image'
    }
    generator = generator_map[category]

    print(f'\nProcessing {category} ({len(files)} images)...')
    success = 0

    for filename in tqdm(files, desc=category):
        src_path = os.path.join(folder, filename)

        try:
            crop = center_crop_and_resize(src_path)
        except Exception as e:
            failures.append({'file': filename, 'category': category, 'error': str(e)})
            continue

        if crop is None:
            failures.append({'file': filename, 'category': category, 'error': 'failed_to_load'})
            continue

        # Save the crop
        out_name = Path(filename).stem + '.png'
        out_path = os.path.join(CROPS_DIR, category, out_name)
        Image.fromarray(crop).save(out_path, 'PNG')

        records.append({
            'filename': out_name,
            'category': category,
            'label': label,
            'generator': generator,
            'crop_path': os.path.join('crops', category, out_name),
        })
        success += 1

    print(f'  → {success}/{len(files)} images processed')

print(f'\n=== Processing Complete ===')
print(f'Total successful crops: {len(records)}')
print(f'Total failures: {len(failures)}')

In [ ]:
# Show failure summary
if failures:
    fail_df = pd.DataFrame(failures)
    print('\n--- Failure Summary ---')
    print(fail_df.groupby(['category', 'error']).size().to_string())

    # Save failure log
    fail_df.to_csv(os.path.join(PROCESSED_DIR, 'failed_loads.csv'), index=False)
    print(f'\nFull failure log saved to: {PROCESSED_DIR}/failed_loads.csv')
else:
    print('No failures — all images loaded and processed successfully!')

## 5. DCT Frequency Representations

Compute the log-DCT representation for each cropped face:
1. Convert to grayscale (Y luminance channel)
2. Apply 2D DCT
3. Log-scale: `log(1 + |F(u,v)|)`
4. Normalise to [0, 1]

These serve as input to the frequency analysis branch (EfficientNet-B0).

In [ ]:
from scipy.fft import dctn

DCT_DIR = os.path.join(PROCESSED_DIR, 'dct')

for name in folders.keys():
    os.makedirs(os.path.join(DCT_DIR, name), exist_ok=True)

def compute_dct(image_array):
    """
    Compute log-DCT representation from an RGB image array.

    Args:
        image_array: np.ndarray (H, W, 3) RGB image
    Returns:
        np.ndarray (H, W) float32, normalised log-DCT
    """
    # Convert to grayscale (luminance)
    gray = cv2.cvtColor(image_array, cv2.COLOR_RGB2GRAY).astype(np.float64)

    # Apply 2D DCT
    dct_coeffs = dctn(gray, norm='ortho')

    # Log-scale to compress dynamic range
    log_dct = np.log(1 + np.abs(dct_coeffs))

    # Normalise to [0, 1]
    dct_min, dct_max = log_dct.min(), log_dct.max()
    if dct_max - dct_min > 1e-8:
        log_dct = (log_dct - dct_min) / (dct_max - dct_min)
    else:
        log_dct = np.zeros_like(log_dct)

    return log_dct.astype(np.float32)

print('DCT function ready.')

In [ ]:
# Compute DCT for all cropped faces
print('Computing DCT representations...')

dct_failures = 0

for i, record in enumerate(tqdm(records, desc='DCT')):
    crop_path = os.path.join(PROCESSED_DIR, record['crop_path'])

    try:
        img = np.array(Image.open(crop_path))
        dct_rep = compute_dct(img)

        # Save as .npy for exact numerical preservation
        dct_filename = Path(record['filename']).stem + '.npy'
        dct_path = os.path.join(DCT_DIR, record['category'], dct_filename)
        np.save(dct_path, dct_rep)

        # Update record with DCT path
        records[i]['dct_path'] = os.path.join('dct', record['category'], dct_filename)

    except Exception as e:
        dct_failures += 1
        records[i]['dct_path'] = None
        print(f'  DCT failed for {record["filename"]}: {e}')

print(f'\nDCT computation complete. Failures: {dct_failures}/{len(records)}')

In [ ]:
# Visualise DCT representations
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Show one example per category: RGB crop (top) and DCT (bottom)
for idx, category in enumerate(folders.keys()):
    cat_records = [r for r in records if r['category'] == category]
    if not cat_records:
        continue

    r = cat_records[0]

    # RGB crop
    crop = np.array(Image.open(os.path.join(PROCESSED_DIR, r['crop_path'])))
    axes[0, idx].imshow(crop)
    axes[0, idx].set_title(f'{category} (RGB)')
    axes[0, idx].axis('off')

    # DCT representation
    if r['dct_path']:
        dct_img = np.load(os.path.join(PROCESSED_DIR, r['dct_path']))
        axes[1, idx].imshow(dct_img, cmap='viridis')
        axes[1, idx].set_title(f'{category} (DCT)')
    axes[1, idx].axis('off')

plt.suptitle('Face Crops (top) and DCT Representations (bottom)', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Train/Validation/Test Split

**Mixed-Generator Strategy:**
All three fake generators (Stable Diffusion, Flux, Z-Image) are mixed across train/val/test so the model can learn each generator's signature.

- **Train (70%):** ~4,200 images — proportional mix of real + SD + Flux + Z-Image
- **Val (15%):** ~900 images — same proportional mix
- **Test (15%):** ~900 images — same proportional mix

Stratified on `generator` to keep the class and generator balance identical across splits.


In [ ]:
from sklearn.model_selection import train_test_split

# Convert records to DataFrame (only those with successful DCT)
manifest_df = pd.DataFrame([r for r in records if r.get('dct_path') is not None])

# Stratified split on `generator` so each split gets proportional real/SD/Flux/Z-Image
# 70% train, 15% val, 15% test
train_df, temp_df = train_test_split(
    manifest_df,
    test_size=0.30,
    random_state=42,
    stratify=manifest_df['generator']
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df['generator']
)

train_df = train_df.copy(); train_df['split'] = 'train'
val_df   = val_df.copy();   val_df['split']   = 'val'
test_df  = test_df.copy();  test_df['split']  = 'test'

df = pd.concat([train_df, val_df, test_df]).reset_index(drop=True)

print('--- Mixed-Generator Split Summary ---')
print(f"Train: {len(train_df):4d}  {train_df['generator'].value_counts().to_dict()}")
print(f"Val:   {len(val_df):4d}  {val_df['generator'].value_counts().to_dict()}")
print(f"Test:  {len(test_df):4d}  {test_df['generator'].value_counts().to_dict()}")
print()
print(f"Train label counts: {train_df['label'].value_counts().to_dict()}")
print(f"Val   label counts: {val_df['label'].value_counts().to_dict()}")
print(f"Test  label counts: {test_df['label'].value_counts().to_dict()}")


In [ ]:
# Detailed breakdowns — use the df we just built, don't rebuild it
print('\n--- Split × Category Breakdown ---')
print(df.groupby(['split', 'category']).size().unstack(fill_value=0).to_string())

print('\n--- Split × Label Breakdown ---')
df['label_name'] = df['label'].map({0: 'real', 1: 'fake'})
print(df.groupby(['split', 'label_name']).size().unstack(fill_value=0).to_string())

## 7. Save Manifest & Final Output

In [ ]:
# Save manifest.csv
manifest_path = os.path.join(PROCESSED_DIR, 'manifest.csv')
df_out = df[['filename', 'category', 'label', 'generator', 'crop_path', 'dct_path', 'split']]
df_out.to_csv(manifest_path, index=False)

print(f'Manifest saved to: {manifest_path}')
print(f'Total records: {len(df_out)}')
print(f'\nFirst 5 rows:')
df_out.head()

In [ ]:
# Verify output directory structure
print('\n--- Output Directory Structure ---')
print(f'{PROCESSED_DIR}/')
for root, dirs, files in os.walk(PROCESSED_DIR):
    level = root.replace(PROCESSED_DIR, '').count(os.sep)
    indent = '  ' * level
    folder_name = os.path.basename(root)
    file_count = len([f for f in files if not f.startswith('.')])
    print(f'{indent}{folder_name}/ ({file_count} files)' if level > 0 else '')

# Total disk usage
total_size = sum(
    os.path.getsize(os.path.join(dirpath, f))
    for dirpath, _, filenames in os.walk(PROCESSED_DIR)
    for f in filenames
)
print(f'\nTotal processed dataset size: {total_size / 1024 / 1024:.1f} MB')

## 8. Quick Sanity Check — Load a Batch

Verify we can load crops + DCT representations as a PyTorch Dataset (preview for Notebook 2).

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class DeepfakeDataset(Dataset):
    """
    Dataset for loading face crops + DCT representations.
    Returns both RGB and DCT inputs for the dual-branch architecture.
    """
    def __init__(self, manifest_df, data_root, split='train', rgb_transform=None):
        self.data_root = data_root
        self.df = manifest_df[manifest_df['split'] == split].reset_index(drop=True)
        self.rgb_transform = rgb_transform or transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Load RGB crop
        rgb_path = os.path.join(self.data_root, row['crop_path'])
        rgb = Image.open(rgb_path).convert('RGB')
        rgb = self.rgb_transform(rgb)

        # Load DCT representation
        dct_path = os.path.join(self.data_root, row['dct_path'])
        dct = np.load(dct_path)
        dct = torch.tensor(dct, dtype=torch.float32).unsqueeze(0)  # (1, H, W)

        label = torch.tensor(row['label'], dtype=torch.float32)

        return rgb, dct, label

# Quick test
manifest_df = pd.read_csv(manifest_path)
test_ds = DeepfakeDataset(manifest_df, PROCESSED_DIR, split='train')

print(f'Train dataset size: {len(test_ds)}')

rgb, dct, label = test_ds[0]
print(f'RGB shape:   {rgb.shape}')       # (3, 224, 224)
print(f'DCT shape:   {dct.shape}')       # (1, 224, 224)
print(f'Label:       {label.item()} ({"fake" if label.item() == 1 else "real"})')

# Test DataLoader
test_loader = DataLoader(test_ds, batch_size=16, shuffle=True, num_workers=2)
batch_rgb, batch_dct, batch_labels = next(iter(test_loader))
print(f'\nBatch RGB shape:    {batch_rgb.shape}')    # (16, 3, 224, 224)
print(f'Batch DCT shape:    {batch_dct.shape}')    # (16, 1, 224, 224)
print(f'Batch labels shape: {batch_labels.shape}')  # (16,)
print(f'Label distribution: {batch_labels.sum().item():.0f} fake, {(1-batch_labels).sum().item():.0f} real')

print('\n✅ Dataset and DataLoader working correctly!')

In [ ]:
print('=' * 50)
print('NOTEBOOK 1 COMPLETE')
print('=' * 50)
print(f'\nProcessed dataset location:\n  {PROCESSED_DIR}')
print(f'\nDataset composition:')
print(f'  Train: {len(train_df)} images')
print(f'  Val:   {len(val_df)} images')
print(f'  Test:  {len(test_df)} images ({len(test_fake)} Flux held-out)')
print(f'\nReady for Notebook 2: Model Training')